# First, let us set up a few dependencies

Don't forget to switch to a GPU-enabled colab runtime!

```
Runtime -> Change Runtime Type -> GPU
```

# Automatic Labelling Using [SAM3](https://ai.meta.com/sam3/)

## Setup

To be able to use this model you would need to acquire access. Check [the model page](https://huggingface.co/facebook/sam3) to request access and confirm you're approved. To run inference, either set your `HF_TOKEN` as an environment variable/secret or log in using the cell below. If you don't already have a token, follow the [instructions](https://huggingface.co/docs/hub/en/security-tokens) on how to create one.

In [ ]:
from huggingface_hub import login
login(new_session=False)

In [ ]:
using_colab = True

In [ ]:
if using_colab:
    import sys
    import torch
    import torchvision
    print("PyTorch version:", torch.__version__)
    print("Torchvision version:", torchvision.__version__)
    print("CUDA is available:", torch.cuda.is_available())
    !{sys.executable} -m pip install opencv-python matplotlib scikit-learn decord numpy==1.26.0

    !git clone https://github.com/facebookresearch/sam3.git /content/sam3

    %cd /content/sam3
    !{sys.executable} -m pip install -e .
    %cd /content

    sam3_repo_root = "/content/sam3"
    if sam3_repo_root not in sys.path:
        sys.path.insert(0, sam3_repo_root)

    if 'sam3' in sys.modules:
        del sys.modules['sam3']

In [ ]:
import os
import shutil

import csv
from pathlib import Path

import torch
import yaml
from PIL import Image
from tqdm import tqdm
from glob import glob

## This mounts your google drive to this notebook. You might have to change the path to fit with your dataset folder inside your drive.

Read the instruction output by the cell bellow carefully!

In [ ]:
# Mount the drive
from google.colab import drive
drive.mount('/content/drive/MyDrive')
DRIVE_PATH = "/content/drive/MyDrive/dt_object_detection_training" #update the path with the location of zipfile

In [ ]:
# Unzip the dataset
DATASET_DIR_NAME = "duckietown_dataset"
DATASET_ZIP_NAME = f"{DATASET_DIR_NAME}.zip"
DATASET_DIR_PATH = os.path.join("/content", DATASET_DIR_NAME)
TRAIN_DIR = "train"
VALIDATION_DIR = "val"
IMAGES_DIR = "images"
LABELS_DIR = "labels"


def show_info(base_path: str):
  for l1 in [TRAIN_DIR, VALIDATION_DIR]:
    for l2 in [IMAGES_DIR, LABELS_DIR]:
      p = os.path.join(base_path, l1, l2)
      print(f"#Files in {l1}/{l2}: {len(os.listdir(p))}")


def unzip_dataset():
  # check zipped file
  zip_path = os.path.join(DRIVE_PATH, DATASET_ZIP_NAME)
  assert os.path.exists(zip_path), f"No zipped dataset found at {zip_path}! Abort!"

  # unzip the data
  print("Unpacking zipped data...")
  shutil.unpack_archive(zip_path, DATASET_DIR_PATH)
  print(f"Zipped dataset unpacked to {DATASET_DIR_PATH}")

  # show some info
  show_info(DATASET_DIR_PATH)


unzip_dataset()

In [ ]:
DATASET_DIR = DATASET_DIR_PATH
TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"
CLASSES_YAML = DATASET_DIR / "classes.yaml"

## Let's create the config file we need for training our duckie detector as well.

In [ ]:
%%writefile /content/duckietown_dataset/classes.yaml

# train and val data as 1) directory: path/images/, 2) file: path/images.txt, or 3) list: [path1/images/, path2/images/]
train:  /content/duckietown_dataset/train
val:    /content/duckietown_dataset/val

# class names
names:
  0: 'yellow rubber duck' # you can try adding more classes if you'd like

# Run Auto Labelling using SAM3

In [ ]:
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

In [ ]:
def load_classes():
    with open(CLASSES_YAML, "r") as f:
        cfg = yaml.safe_load(f)
    items = sorted(cfg["names"].items(), key=lambda kv: int(kv[0]))
    return [name for _, name in items]


def xyxy_to_yolo_line(bbox, w, h, class_id):
    x1, y1, x2, y2 = bbox
    cx = (x1 + x2) / 2.0 / w
    cy = (y1 + y2) / 2.0 / h
    bw = (x2 - x1) / w
    bh = (y2 - y1) / h
    return f"{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"


def init_sam3(device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    classes = load_classes()
    bpe_path = Path("/content/sam3/assets/bpe_simple_vocab_16e6.txt.gz")
    model = build_sam3_image_model(bpe_path=str(bpe_path))
    model.to(device)
    model.eval()
    processor = Sam3Processor(model, confidence_threshold=0.3, device=device)
    return processor, classes


def detect_sam3(img_path, state):
    processor, classes = state
    image = Image.open(img_path).convert("RGB")
    w, h = image.size
    with torch.no_grad():
        st = processor.set_image(image)
        detections = []
        for class_id, prompt in enumerate(classes):
            out = processor.set_text_prompt(state=st, prompt=prompt)
            boxes = out.get("boxes")
            scores = out.get("scores")
            if boxes is None or scores is None:
                continue
            for b, s in zip(boxes, scores):
                s = float(s)
                if s < processor.confidence_threshold:
                    continue
                detections.append(
                    {
                        "bbox": b.tolist(),
                        "score": s,
                        "class_id": int(class_id),
                    }
                )
    return detections, (w, h)


def auto_label_folder(data_dir: Path, split: str = "train", backend: str = "sam3"):
    if split == "train":
        split_dir = TRAIN_DIR
    elif split == "val":
        split_dir = VAL_DIR
    else:
        raise ValueError("split must be 'train' or 'val'")

    images_out = split_dir / "images"
    labels_out = split_dir / "labels"
    images_out.mkdir(parents=True, exist_ok=True)
    labels_out.mkdir(parents=True, exist_ok=True)

    csv_path = DATASET_DIR / f"autolabel_{backend}_{split}.csv"
    csv_file = open(csv_path, "w", newline="")
    writer = csv.writer(csv_file)
    writer.writerow(["backend", "image_path", "class_id", "score", "x1", "y1", "x2", "y2"])

    if backend != "sam3":
        raise ValueError(f"unsupported backend: {backend}")

    state = init_sam3()

    image_paths = glob(f"{images_out}/**/*", recursive=True)
    print(len(image_paths))

    for img_path in tqdm(image_paths):
        dets, (w, h) = detect_sam3(img_path, state)
        out_img = images_out / Path(img_path).name

        yolo_lines = []
        for d in dets:
            x1, y1, x2, y2 = d["bbox"]
            score = d["score"]
            class_id = d["class_id"]
            writer.writerow(
                ["sam3", str(out_img), class_id, score, x1, y1, x2, y2]
            )
            yolo_lines.append(
                xyxy_to_yolo_line(d["bbox"], w, h, class_id)
            )

        label_path = labels_out / (Path(img_path).stem + ".txt")
        if yolo_lines:
            label_path.write_text("\n".join(yolo_lines))
        else:
            if label_path.exists():
                label_path.unlink()

    csv_file.close()

In [ ]:
import matplotlib.pyplot as plt

def visualize_labels_for_image(img_path: Path, labels_dir: Path, class_names=None, save_to: Path | None = None):
    img_path = Path(img_path)
    label_path = labels_dir / (img_path.stem + ".txt")

    if not label_path.exists():
        print(f"No label file found for {img_path.name}")
        return

    image = Image.open(img_path).convert("RGB")
    w, h = image.size

    with open(label_path, "r") as f:
        lines = [ln.strip() for ln in f.readlines() if ln.strip()]

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image)
    ax.axis("off")

    for line in lines:
        parts = line.split()
        if len(parts) != 5:
            continue
        class_id = int(parts[0])
        cx, cy, bw, bh = map(float, parts[1:])

        box_w = bw * w
        box_h = bh * h
        x1 = cx * w - box_w / 2
        y1 = cy * h - box_h / 2

        rect = plt.Rectangle(
            (x1, y1),
            box_w,
            box_h,
            fill=False,
            linewidth=2,
        )
        ax.add_patch(rect)

        label = str(class_id)
        if class_names is not None and 0 <= class_id < len(class_names):
            label = class_names[class_id]

        ax.text(
            x1,
            y1 - 2,
            label,
            fontsize=10,
            bbox=dict(facecolor="black", alpha=0.5),
            color="white",
        )

    if save_to is not None:
        save_to = Path(save_to)
        save_to.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_to, bbox_inches="tight", pad_inches=0)
        plt.close(fig)
    else:
        plt.show()

### Run SAM3 on your training set!

In [ ]:
auto_label_folder(DATASET_DIR, "train")

### Run SAM3 on your validation set

In [ ]:
auto_label_folder(DATASET_DIR, "val")

### Lets visualize some of the predicted detections

In [ ]:
train_images = TRAIN_DIR / "images"
train_labels = TRAIN_DIR / "labels"

img = next(train_images.glob("*.png"))

visualize_labels_for_image(
    img_path=img,
    labels_dir=train_labels,
    class_names=load_classes(),
)

## Training Setup

In [ ]:
!pip install -q ultralytics numpy

# And we're ready to train! This step will take about 5 minutes.

Notice that we're only training for 10 epochs. That's probably not enough!

You can modify the config file in `classes.yaml` to update the class lists. If you have done so previously for the autolabelling or you can keep it as is.

In [ ]:
from ultralytics import YOLO

# training from scratch - you can play around with different yolo models if you wish
model = YOLO("yolo11n.yaml")

runs_root = DATASET_DIR / "runs"
run_name = "duckietown_detection"

model.train(
    data=str(CLASSES_YAML),
    epochs=10,
    imgsz=(480,640),
    batch=16,
    workers=2,
    project=str(runs_root),
    name=run_name,
)

### You can check the `duckietwon_dataset/runs/<run_name>` to see the confusion matrix, percision/recall/ROC plots, and some sample visualization.

## Export your best model to ONNX

In [ ]:
import os
import numpy as np

all_exps = os.listdir(f"{DATASET_DIR}/runs")
all_exps_filtered = map(lambda x: int(x.replace("duckietown_dataset", "1")), filter(lambda x: x.startswith("exp"), all_exps))
all_exps_filtered = np.array(list(all_exps))
latest_exp_index = np.argmax(all_exps)
latest_exp = all_exps[latest_exp_index]
print(f"Latest exp is {latest_exp}")

In [ ]:
run_dir = DATASET_DIR / "runs" / f"{latest_exp}" # you may need to update this based on you run_id
best = run_dir / "weights" / "best.pt"
model_path = DATASET_DIR / "weights" / "best.onnx"

if not best.exists():
    raise FileNotFoundError(f"Could not find best.pt at: {best}")

print(f"Using checkpoint: {best}")
model = YOLO(str(best))

onnx_tmp = run_dir / "weights" / "best.onnx"
print(f"Exporting ONNX to: {onnx_tmp}")

model.export(
    format="onnx",
    opset=18,
    imgsz=(480,640), # (height, width)
    simplify=True,
    dynamic=False,
    nms=True,
    half=True,
    batch=1,
    optimize=False,
    )

model_path.parent.mkdir(parents=True, exist_ok=True)

model_path.write_bytes(onnx_tmp.read_bytes())

print(f"Copied ONNX model to: {model_path}")
print("Export finished successfully.")

# Next, you can download your model and place it in your local `lx-object-dection/assets/` folder to later test the agents behavior!

Make sure you have downloaded the `best.onnx` file before moving to the next notebook.

# Done!

We're done training! You can now close this tab and go back to the `Training` notebook